## 自定义中间件

#### 加载环境

In [ ]:
from dotenv import load_dotenv

# 读取 .env 配置
load_dotenv(override=True)

#### 基于装饰器实现

底层也是基于 AgentMiddleware 实现

In [ ]:
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_model, after_model, before_agent, after_agent
from langchain.agents import AgentState
from typing import Any

@before_model
def before_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> before model <----"
    return None

@after_model
def after_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> after model <----"
    return None

@before_agent
def before_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "，忽略后面的信息：----> before agent <----"
    return None

@after_agent
def after_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> after agent <----"
    return None


In [ ]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model='deepseek:deepseek-flash',
    middleware=[
        before_model_middleware,
        after_model_middleware,
        before_agent_middleware,
        after_agent_middleware
    ]
)

resp = agent.invoke({'messages': HumanMessage("你好")})
for msg in resp["messages"]:
    msg.pretty_print()

#### 基于类实现

In [ ]:
from langgraph.runtime import Runtime
from langgraph.typing import StateT
from langchain.agents.middleware import AgentMiddleware
from typing import Any

class MyMiddleware(AgentMiddleware):

    def before_agent(self, state: StateT, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "，忽略后面的信息：----> before agent <----"
        return None

    def after_agent(self, state: StateT, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> after agent <----"
        return None

    def before_model(self, state: StateT, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> before model <----"
        return None

    def after_model(self, state: StateT, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> after model <----"
        return None


In [ ]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model='deepseek:deepseek-flash',
    middleware=[MyMiddleware()]
)

resp = agent.invoke({'messages': HumanMessage("你好")})
for msg in resp["messages"]:
    msg.pretty_print()

##### can_jump_to

In [ ]:
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import AgentState, before_model
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.runtime import Runtime
from rich.pretty import pprint


@before_model(can_jump_to=["end"])
def check_input(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:

    message = state["messages"][-1]

    # 20 个字符只是演示阈值
    if isinstance(message, HumanMessage) and len(message.text) > 20:
        print("输入过长，跳过模型调用")

        return {
            # 由程序直接生成回答，追加到消息列表
            "messages": [
                AIMessage(content="输入超过20个字符，请精简后重试。")
            ],
            # 跳转到结束流程
            "jump_to": "end",
        }

    # 不修改状态，继续正常调用模型
    return None


agent = create_agent(
    model="deepseek:deepseek-flash",
    middleware=[check_input],
)

resp = agent.invoke({"messages": HumanMessage("这是一段很长的输入。" * 5)})
pprint(resp)